In [1]:
import sys
from pathlib import Path
import json
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pprint import pp

In [2]:
ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from src.config import BANQUES
from src.extraction import save_pages
DATA = ROOT_DIR / 'data'

In [3]:
with (DATA/'interim/ca_2025.jsonl').open(encoding='UTF-8') as pdf :
    pages_ca = [Document(**json.loads(page)) for page in pdf]

In [4]:
len(pages_ca[46].page_content),pages_ca[46].metadata

(5431,
 {'source': 'ca_2025.pdf',
  'banque': 'Credit Agricole',
  'annee': 2025,
  'langue': 'fr',
  'page': 47})

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,    # 375~430 tokens (/4 en anglais /3.5 en fr) 
    chunk_overlap=200,  #
    keep_separator="end",
    separators=['\n\n', #defaut
                '\n',   #defaut
                '. ',
                '; ',
                ', ',
                ' ',    #defaut
                '']     #defaut
    )

In [6]:
chunks_ca = splitter.split_documents([pages_ca[46]]) # prend seulement une liste de documents en entreem renvoie [doc]
len(chunks_ca)

6

In [7]:
max([len(c.page_content) for c in chunks_ca]),min([len(c.page_content) for c in chunks_ca]),next((i for i,c in enumerate(chunks_ca) if c.metadata['page']!=47),None)

(1439, 176, None)

In [8]:
for d in chunks_ca:
    pp(d.page_content)

('###### Frontière des contrats \n'
 '\n'
 "L'évaluation d'un groupe de contrats intègre tous les flux de trésorerie "
 'futurs compris dans le périmètre (la « frontière ») de chacun des contrats '
 "du groupe, c'est-à-dire tous les flux de trésorerie futurs qui découlent de "
 'droits et obligations substantiels qui existent au cours de la période de '
 'reporting dans laquelle le Groupe peut contraindre le titulaire de police à '
 'payer les primes ou dans laquelle il a une obligation substantielle de lui '
 "fournir des services prévus au contrat d'assurance. \n"
 '\n'
 'Une telle obligation substantielle cesse notamment lorsque le Groupe a la '
 'capacité pratique de réévaluer les risques posés spécifiquement par le '
 'titulaire de police et peut en conséquence fixer un prix ou un niveau de '
 'prestations qui reflète intégralement ces risques réévalués.')
('La détermination de la frontière des contrats requiert de faire preuve de '
 'jugement et de prendre en compte les droits et

In [9]:
with ( DATA/'interim/bnpparibas_2025.jsonl').open(encoding="utf-8") as f:
    pages_bnp = [Document(**json.loads(lines) ) for lines in f]

In [10]:
page_bnp = next(p for p in pages_bnp if p.page_content.count('\n|')>30 and "millions d" in p.page_content) 
# on cherche un tableau pour voir comment il sera decoupe
page_bnp.metadata

{'source': 'bnpparibas_2025.pdf',
 'banque': 'BNP Paribas',
 'annee': 2025,
 'langue': 'fr',
 'page': 118}

In [11]:
pp(page_bnp.page_content)

('GOUVERNEMENT D’ENTREPRISE ET CONTRÔLE INTERNE \n'
 '\n'
 'Rapport sur le gouvernement d’entreprise \n'
 '\n'
 'Le tableau ci‑dessous présente les multiples de rémunération et leurs '
 'évolutions pour chaque dirigeant mandataire social. \n'
 '\n'
 '||2021|2022 (1)|2023 (2)|2024|Année 2025|\n'
 '|---|---|---|---|---|---|\n'
 '|Performance de la société||||||\n'
 '|Résultat net avant impôt (en millions d’euros)|13 637|13 214|11 725|16 '
 '188|17 065|\n'
 '|Évolution N/N‑1|39 %|6 %|- 11 %|38 %|5 %|\n'
 '|Résultat d’exploitation (en millions d’euros)|12 199|12 564|11 236|15 '
 '437|16 296|\n'
 '|Évolution N/N‑1|46 %|13 %|- 11 %|37 %|6 %|\n'
 '|Bénéfce net par action (en euros)|7,26|7,80|8,58|9,57|10,29|\n'
 '|Évolution N/N‑1|37 %|7 %|10 %|12 %|8 %|\n'
 '|Rémunération des salariés(en milliers d’euros)||||||\n'
 '|Rémunération moyenne (en milliers d’euros)|93|96|99|101|102|\n'
 '|Évolution N/N‑1|6 %|3 %|2 %|2 %|1 %|\n'
 '|Rémunération médiane (en milliers d’euros)|59|62|66|67|68|\n'
 '|Évo

In [12]:
chunks_bnp = splitter.split_documents([page_bnp])
len(chunks_bnp)
chunks_bnp

[Document(metadata={'source': 'bnpparibas_2025.pdf', 'banque': 'BNP Paribas', 'annee': 2025, 'langue': 'fr', 'page': 118}, page_content='GOUVERNEMENT D’ENTREPRISE ET CONTRÔLE INTERNE \n\nRapport sur le gouvernement d’entreprise \n\nLe tableau ci‑dessous présente les multiples de rémunération et leurs évolutions pour chaque dirigeant mandataire social.'),
 Document(metadata={'source': 'bnpparibas_2025.pdf', 'banque': 'BNP Paribas', 'annee': 2025, 'langue': 'fr', 'page': 118}, page_content='||2021|2022 (1)|2023 (2)|2024|Année 2025|\n|---|---|---|---|---|---|\n|Performance de la société||||||\n|Résultat net avant impôt (en millions d’euros)|13 637|13 214|11 725|16 188|17 065|\n|Évolution N/N‑1|39 %|6 %|- 11 %|38 %|5 %|\n|Résultat d’exploitation (en millions d’euros)|12 199|12 564|11 236|15 437|16 296|\n|Évolution N/N‑1|46 %|13 %|- 11 %|37 %|6 %|\n|Bénéfce net par action (en euros)|7,26|7,80|8,58|9,57|10,29|\n|Évolution N/N‑1|37 %|7 %|10 %|12 %|8 %|\n|Rémunération des salariés(en milliers 

In [13]:
for i,c in enumerate(chunks_bnp):
    pp(f"chunk {i+1} ({len(c.page_content)} carac)")
    pp(c.page_content)
    print()

'chunk 1 (212 carac)'
('GOUVERNEMENT D’ENTREPRISE ET CONTRÔLE INTERNE \n'
 '\n'
 'Rapport sur le gouvernement d’entreprise \n'
 '\n'
 'Le tableau ci‑dessous présente les multiples de rémunération et leurs '
 'évolutions pour chaque dirigeant mandataire social.')

'chunk 2 (1496 carac)'
('||2021|2022 (1)|2023 (2)|2024|Année 2025|\n'
 '|---|---|---|---|---|---|\n'
 '|Performance de la société||||||\n'
 '|Résultat net avant impôt (en millions d’euros)|13 637|13 214|11 725|16 '
 '188|17 065|\n'
 '|Évolution N/N‑1|39 %|6 %|- 11 %|38 %|5 %|\n'
 '|Résultat d’exploitation (en millions d’euros)|12 199|12 564|11 236|15 '
 '437|16 296|\n'
 '|Évolution N/N‑1|46 %|13 %|- 11 %|37 %|6 %|\n'
 '|Bénéfce net par action (en euros)|7,26|7,80|8,58|9,57|10,29|\n'
 '|Évolution N/N‑1|37 %|7 %|10 %|12 %|8 %|\n'
 '|Rémunération des salariés(en milliers d’euros)||||||\n'
 '|Rémunération moyenne (en milliers d’euros)|93|96|99|101|102|\n'
 '|Évolution N/N‑1|6 %|3 %|2 %|2 %|1 %|\n'
 '|Rémunération médiane (en milli

In [14]:
def read_pages(jsonl_path):
    """Renvois liste de documents lu d'un jsonl """
    with open(jsonl_path, 'r',encoding='utf-8') as f:
        pages = [Document(**json.loads(line)) for line in f]
    return pages

In [15]:
chunks_bnp[0].metadata['chunk_id'] = 1

In [16]:
chunks_bnp[0].metadata['source'].rsplit('.',1)[0] + '_' + str(1)

'bnpparibas_2025_1'

In [17]:
stem = chunks_bnp[0].metadata['source'].rsplit('.',1)[0]
page_number = chunks_bnp[0].metadata['page']
f'{stem}_p{page_number}_c{1}'

'bnpparibas_2025_p118_c1'

In [18]:
def chunk_pages(pages, splitter):
    chunk_list = splitter.split_documents(pages)
    iD = 0
    page_precedent = None
    for c in chunk_list:
        source = c.metadata['source'].rsplit('.',1)[0]
        page_number = c.metadata['page']
        if page_precedent == page_number:
            iD +=1
        else: 
            iD =0
            page_precedent = page_number
        c.metadata['chunk_id'] = f'{source}_p{page_number}_c{iD}'
    return chunk_list

In [48]:
def dump_chunk(in_path, out_path,splitter):
    chunks = []
    for jsonl in sorted(Path(in_path).glob('*.jsonl')):
        c = chunk_pages(read_pages(jsonl),splitter)
        chunks.extend(c)
        print(f'{jsonl.name} lu, {len(c)} chunks crees')
    tailles = [len(c.page_content) for c in chunks]
    obs = f'Au total, {len(chunks)} chunks crees, max, min et mediane des chunks est {max(tailles), min(tailles), sorted(tailles)[len(tailles)//2]}'
    print(obs)
    ob1 = sum(len(c.page_content)<50 for c in chunks)
    ob2 = sum(len(c.page_content)<100 for c in chunks)
    ob3 = sum(len(c.page_content)<200 for c in chunks)
    print(f'''On observe:
    -{ob1} chunks a moins de 50 caracteres soit {round(ob1/len(chunks)*100,3)}% des chunks
    -{ob2} chunks a moins de 100 caracteres soit {round(ob2/len(chunks)*100,3)}% des chunks
    -{ob3} chunks a moins de 200 caracteres soit {round(ob3/len(chunks)*100,3)}% des chunks''')
    print('Un chunk au hasard :')
    pp(chunks[250].metadata)
    save_pages(chunks, Path(out_path))

In [49]:
dump_chunk(DATA/'interim', DATA/'processed/chunks.jsonl', splitter)

bnpparibas_2025.jsonl lu, 3445 chunks crees
ca_2025.jsonl lu, 776 chunks crees
ubs_2025.jsonl lu, 1619 chunks crees
Au total, 5840 chunks crees, max, min et mediane des chunks est (1500, 1, 1211)
On observe:
    -32 chunks a moins de 50 caracteres soit 0.548% des chunks
    -142 chunks a moins de 100 caracteres soit 2.432% des chunks
    -352 chunks a moins de 200 caracteres soit 6.027% des chunks
Un chunk au hasard :
{'source': 'bnpparibas_2025.pdf',
 'banque': 'BNP Paribas',
 'annee': 2025,
 'langue': 'fr',
 'page': 72,
 'chunk_id': 'bnpparibas_2025_p72_c2'}
